# 🧪 Thực Nghiệm 1: Đo Đạc Hệ Số Lipschitz Hàm Threshold Reward
### **Vanilla LiDAR vs. RS-LiDAR (Randomized Smoothing)** trên GPU T4/A100

Notebook này được thiết kế độc lập để kiểm chứng **Định lý Giảm Lipschitz** trên hàm bước nhảy (Threshold/Binary Reward), sử dụng **OWL-ViT** cho bài toán Zero-shot Object Counting.
- **Mục tiêu**: Hàm Threshold Reward gốc trả về giá trị rời rạc `{0, 1}` nên đạo hàm tại điểm đứt gãy tiến tới $\infty$. Vanilla LiDAR sẽ bị "mù" gradient. RS-LiDAR sử dụng nhiễu Monte Carlo để làm mượt hàm này, biến nó thành một hàm liên tục có độ dốc bị chặn (Lipschitz-continuous).
- **Tính đối xứng nghiêm ngặt**: Hàm smoothed reward $R_{\sigma_2}$ được tính độc lập cho **CẢ ẢNH SẠCH $x_{\text{clean}}$ VÀ ẢNH BIẾN DẠNG $x_{\text{pert}}$**.
- **Công cụ đánh giá**: Chạy trực tiếp qua script `test_threshold_reward.py`, hỗ trợ chạy đa tiến trình Sharding trên Kaggle/Colab, xuất bộ đồ thị 3-panel chuẩn báo cáo khoa học.

## 1. Kiểm Tra Phần Cứng GPU & Cấu Hình Bộ Nhớ
Đảm bảo bạn đã chọn GPU (T4, V100, hoặc A100).

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings.")

# Chống phân mảnh bộ nhớ VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi

## 2. Đồng Bộ Mã Nguồn Repo & Cài Đặt Môi Trường

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, urllib.request

# 1. Đồng bộ repo RS-LiDAR
REPO_DIR = "/content/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

os.chdir(REPO_DIR)
%cd {REPO_DIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài Đặt các thư viện thiết yếu
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q matplotlib tqdm scipy seaborn pandas tabulate

# 3. Tiện ích chạy song song đa GPU
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    rc0 = p0.wait()
    rc1 = p1.wait()
    if rc0 != 0 or rc1 != 0:
        raise RuntimeError(f"Tiến trình GPU thất bại (GPU 0: {rc0}, GPU 1: {rc1})")

print("\n✅ Môi trường thực nghiệm đo Lipschitz đã sẵn sàng 100%!")

## 3. 📁 Gắn Kết Google Drive Cá Nhân (Tự Động Lưu Kết Quả Chống Mất Dữ Liệu)

In [ ]:
# @title 📁 Gắn kết Google Drive & Thiết lập thư mục lưu trữ
import os
from google.colab import drive

try:
    drive.mount('/content/drive')
    TEST_OUTPUT_DIR = '/content/drive/MyDrive/RS-LiDAR/results/New_Rewards/Threshold_Rewards'
    os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
    print(f"\n✅ Đã gắn kết Google Drive thành công!")
    print(f"📁 Thư mục lưu kết quả trực tiếp: {TEST_OUTPUT_DIR}")
except Exception as e:
    print(f"⚠️ Không thể mount Google Drive: {e}. Kết quả sẽ lưu cục bộ.")
    TEST_OUTPUT_DIR = "/content/RS-LiDAR/results/New_Rewards/Threshold_Rewards"
    os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)


## 3. Cấu Hình Tham Số Thực Nghiệm (Threshold Reward)
Các tham số định cấu hình cho script đo đạc `test_threshold_reward.py`.

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH THAM SỐ (TÙY CHỈNH TẠI ĐÂY)
# ==============================================================================
NUM_PARTICLES = 10                  # K = 10 hạt (số ảnh sạch sinh ra để đánh giá)
SIGMA1 = 0.1                        # Độ lệch chuẩn vi nhiễu tạo x_pert
PERTURBATION_TYPE = "gaussian"      # Kiểu biến dạng: "gaussian"
SIGMA2 = 1.0                        # Bán kính làm mịn RS-LiDAR chính
SIGMA2_SWEEP = "0.0,0.1,0.25,0.5,1.0" # Dải quét sigma2
NUM_MC_SAMPLES = 4                  # M = 4 mẫu Monte Carlo tính kỳ vọng RS-LiDAR
USE_CRN = True                      # Coupled Noise triệt tiêu phương sai Monte Carlo
OVERWRITE = True                    # Đặt True để xóa cache, chạy lại từ đầu
# ==============================================================================

print(f"📊 Số lượng: {NUM_PARTICLES} hạt/prompt")
print(f"🔬 Nhiễu biến dạng sigma1 = {SIGMA1} ({PERTURBATION_TYPE}) | Bán kính làm mịn RS-LiDAR sigma2 = {SIGMA2}")
print(f"📈 Dải khảo sát sigma2: {{{SIGMA2_SWEEP}}} | Monte Carlo M = {NUM_MC_SAMPLES} | CRN = {USE_CRN}")
print(f"📁 Thư mục lưu kết quả: {TEST_OUTPUT_DIR}")

## 4. Thực Thi Đo Lipschitz Song Song (Tự động tải Model & Run)
Script `test_threshold_reward.py` sẽ load Stable Diffusion 1.5 và OWL-ViT, tự động tạo ảnh và nhiễu, chấm điểm Threshold Reward, sau đó lưu các hệ số độ dốc.

In [ ]:
# 1. Dọn dẹp tiến trình cũ
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
!pkill -f "test_threshold_reward.py" || true

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
ov_flag = "--overwrite" if OVERWRITE else ""
crn_flag = "--use_crn" if USE_CRN else "--no_crn"

print(f"\n🚀 Bắt đầu đo Lipschitz trên Threshold Reward với {n_gpus} GPU...\n")

if n_gpus >= 2:
    print("⚡ [2 GPU] Đang chạy song song trên GPU 0 (Shard 0) và GPU 1 (Shard 1)...")
    cmd0 = f"""python -u test_threshold_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 0 --gpu_id 0 {crn_flag} {ov_flag}"""

    cmd1 = f"""python -u test_threshold_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 1 --gpu_id 1 {crn_flag} {ov_flag}"""
    run_commands_parallel(cmd0, cmd1)
else:
    print("⚡ [1 GPU] Đang chạy đo trên 1 GPU...")
    !python -u test_threshold_reward.py \
        --num_particles {NUM_PARTICLES} \
        --sigma1 {SIGMA1} \
        --perturbation_type {PERTURBATION_TYPE} \
        --sigma2 {SIGMA2} \
        --sigma2_list "{SIGMA2_SWEEP}" \
        --num_mc_samples {NUM_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" {crn_flag} {ov_flag}

print("\n✅ HOÀN THÀNH ĐO ĐẠC! Tiến hành gộp shard...")

## 5. Tự Động Gộp Kết Quả Shards & Đồ Thị Hóa (3-Panel Plot)

In [ ]:
print("🔄 Đang gộp kết quả và vẽ biểu đồ 3-Panel...")
!python -u test_threshold_reward.py --merge_shards --output_dir "{TEST_OUTPUT_DIR}" --sigma2 {SIGMA2}
print("✅ Hoàn tất gộp dữ liệu!")

## 6. Hiển Thị Báo Cáo
Biểu đồ 3-Panel dưới đây minh chứng hàm bước nhảy Threshold đã bị san phẳng và độ dốc (Lipschitz) đã giảm đáng kể.

In [ ]:
from IPython.display import Image, display
import glob

plot_files = sorted(glob.glob(f"{TEST_OUTPUT_DIR}/*.png"))
for pf in plot_files:
    print(f"\n📈 Đồ thị báo cáo: {os.path.basename(pf)}")
    display(Image(filename=pf))

## 7. Đóng Gói Toàn Bộ Kết Quả (Tự động đẩy lên Google Drive)

In [ ]:
zip_path = "/content/threshold_reward_results.zip"
print(f"📦 Đang nén toàn bộ kết quả vào {zip_path}...")

!zip -r -q {zip_path} {TEST_OUTPUT_DIR}

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print("👉 Tải file này từ tab Files bên trái Colab, hoặc vào Drive của bạn để lấy.")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")
